# Forelesning 7: Numerisk integrasjon

<div class="bilde-hoyre" style="width:50%">

![Kromatogram med integrerte topper](bilder/kromatogram_areal.png)

</div>

I dag skal vi

- tilnærme et integral som en sum av små arealer: rektangler og trapeser
- undersøke hvor raskt metodene **konvergerer** når vi bruker flere intervaller
- bruke ferdige funksjoner i SciPy
- integrere topper i et kromatogram

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate

## Repetisjon

<div class="oppgave">
Lag en funksjon som deriverer med framoverdifferansen, og en funksjon som bruker Newtons metode til å finne nullpunktet til en funksjon. Bruk tilnærmingen til den deriverte istedenfor den analytisk deriverte i Newtons metode og løs likningen $e^{-t} + t + 5 - \ln(0.006t + 1) = t^{0.3} + 10$.
</div>

## Del 1: Rektangelmetoden

<div class="viktig">

Vi deler intervallet $[a, b]$ i $n$ figurer med bredde

$$h = \frac{b - a}{n}$$

og tilnærmer hver figur med et rektangel. Integralet er summen av arealene: $\int_a^b f(x)\,dx \approx \sum f(x_k) \cdot h$

</div>

<div class="bilde-midt" style="width:60%">

![Venstretilnærming med rektangler](bilder/venstretilnaerming.png)

</div>

### Det hele handler om tilnærminger

<div class="bilde-hoyre" style="width:45%">

![Sum og integral](bilder/sum_vs_integral.png)

</div>

| Når jeg skreller poteter | Når mora mi skreller poteter |
|:---:|:---:|
| $\displaystyle\sum_{k=1}^{n} f(x_k)\cdot\Delta x$ | $\displaystyle\int f(x)\,dx$ |

Summen er en tilnærming til integralet. Jo smalere rektangler, jo mindre «skrell» går til spille.

### Venstretilnærming
Vi integrerer $f(x) = \cos x + 2$ fra 2 til 12. Den eksakte verdien er $\left[\sin x + 2x\right]_2^{12} \approx 18{,}554$.

<div class="oppgave">

**Prøv selv:** Vi pakker løkka inn i en funksjon med et ekstra argument `position`, som er 0 for venstre kant, 1 for høyre kant og 0,5 for midtpunktet. Fyll inn startverdien til `x`. Hvilken metode er best?

</div>

In [ ]:
def rectangle(f, a, b, n, position=0):
    h = (b - a) / n
    area = 0
    x = ??
    for k in range(n):
        area = area + f(x) * h
        x = x + h
    return area

exact = (np.sin(12) + 2*12) - (np.sin(2) + 2*2)

print("Left:    ", rectangle(f, 2, 12, 10, 0))
print("Right:   ", rectangle(f, 2, 12, 10, 1))
print("Midpoint:", rectangle(f, 2, 12, 10, 0.5))
print("Exact:   ", exact)

<div class="bilde-midt" style="width:95%">

![Venstre-, høyre- og midtpunktstilnærming](bilder/rektangelvarianter.png)

</div>

<div class="diskuter">

**Diskuter:** Se på figuren. Hvorfor blir midtpunktstilnærmingen så mye bedre? Hvilke funksjoner gir den et eksakt svar for?

</div>

## Del 2: Konvergens

Et numerisk svar bør ikke vurderes ut fra ett enkelt valg av $n$. Vi dobler antall intervaller og ser hvordan feilen endrer seg.

<div class="diskuter">

**Tipp først:** Hva skjer med feilen når vi dobler $n$?

</div>

In [ ]:
for n in [10, 20, 40, 80, 160]:
    error_left = abs(rectangle(f, 2, 12, n, 0) - exact)
    error_mid = abs(rectangle(f, 2, 12, n, 0.5) - exact)
    print(f"n = {n:4d}   left: {error_left:.2e}   midpoint: {error_mid:.2e}")

<div class="viktig">

Når vi dobler $n$, blir feilen omtrent

- **halvert** for venstre- og høyretilnærmingen (feilen er proporsjonal med $1/n$)
- **fjerdedelt** for midtpunktstilnærmingen (feilen er proporsjonal med $1/n^2$)

Kontroller alltid et numerisk integral ved å øke $n$ og se om svaret stabiliserer seg.

</div>

## Del 3: Trapesmetoden

<div class="bilde-hoyre" style="width:22%">

![It's a trapezoidal rule](bilder/trapezoidal_rule_meme.jpg)

</div>

Rektanglene har en vannrett topp. Trekker vi i stedet en rett linje mellom funksjonsverdiene i de to endepunktene, får vi et **trapes**.

<div class="viktig">

Arealet av ett trapes med bredde $h$:

$$A_k = \frac{f(x_k) + f(x_{k+1})}{2}\cdot h$$

</div>

<div class="bilde-midt" style="width:50%">

![Rektangler og trapeser](bilder/integrasjonsmetoder.png)

</div>

<div class="oppgave">

**Prøv selv:** Kopier løkka fra `rectangle`, og endre den slik at den summerer trapeser i stedet for rektangler. Test på $f(x) = x^3$ fra 0 til 5, som har eksakt verdi 156,25.

</div>

<div class="viktig">

**Mønsteret:** Rektangler bruker en **konstant**, trapeser en **rett linje** og **Simpsons metode** et **andregradspolynom** mellom punktene.

</div>

<div class="bilde-midt" style="width:55%">

![Simpson's rule](bilder/simpsons_rule.png)

</div>

## Del 4: SciPy

<div class="bilde-hoyre" style="width:22%">

![Halvt semester på noe som fantes som bibliotek](bilder/bibliotek_meme.png)

</div>

<div class="viktig">

| Funksjon | Bruk |
|---|---|
| `integrate.trapezoid(y, x)` | måledata: arrayer med x- og y-verdier |
| `integrate.simpson(y, x=x)` | måledata, Simpsons metode |
| `integrate.quad(f, a, b)` | en funksjon vi kan evaluere hvor vi vil |

Eldre kode bruker navnene `trapz` og `simps`, som er fjernet i nyere SciPy.

</div>

### Integrasjon av måledata

En reaksjon utvikler CO₂-gass. En gassmåler registrerer hvor mange mL gass som strømmer ut per sekund, hvert 10. sekund. Det totale gassvolumet er arealet under kurven:

$$V = \int_0^{120} \text{gassfluks}\,dt$$

In [ ]:
time = np.array([0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120])            # s
flow = np.array([0.00, 0.42, 0.65, 0.71, 0.66, 0.55, 0.44, 0.34, 0.26, 0.19, 0.14, 0.10, 0.07])   # mL/s

plt.plot(time, flow, "o-")
plt.xlabel("Time (s)")
plt.ylabel("Gas flow (mL/s)")
plt.show()

<div class="viktig">

Med måledata er steglengden bestemt av hvor ofte vi måler. Enheten til integralet er enheten til $y$ ganger enheten til $x$: mL/s · s = mL.

</div>

## Del 5: Areal under kromatografiske topper

Vi lager et kunstig kromatogram med to topper. Signalet består nå av **målepunkter**, ikke en formel.

In [ ]:
time = np.linspace(0, 10, 501)   # min
peak_1 = 1.2 * np.exp(-0.5 * ((time - 3.0) / 0.35)**2)
peak_2 = 0.8 * np.exp(-0.5 * ((time - 6.5) / 0.50)**2)
signal = peak_1 + peak_2

plt.plot(time, signal)
plt.xlabel("Retention time (min)")
plt.ylabel("Signal (a.u.)")
plt.show()

<div class="viktig">
Topp 1 er 50 % høyere enn topp 2, men arealene er nesten like, fordi topp 2 er bredere. Stoffmengden henger sammen med **arealet**, ikke høyden.
</div>

__Oppgaver__:
1. Integrer $f(x)=x^2-2x+4$ fra 2 til 8 med venstre-, høyre- og midtpunktstilnærming og trapesmetoden. Bruk først $n=10$ og deretter $n=100$, og sammenlikn med den analytiske verdien.
2. Plott den absolutte feilen som funksjon av $n$ for venstre-, midtpunkts- og trapesmetoden med logaritmiske akser. Hva forteller stigningen til linjene?
3. Et kalorimeter registrerer varmeeffekt $P(t)$ i watt hvert sekund. Forklar hvorfor $\int P(t)\,dt$ gir varme, lag et kunstig datasett og integrer det. Hvilken enhet får svaret?
4. To NMR-signaler har arealene 3,02 og 1,01. Hva kan arealforholdet tyde på om det relative antallet hydrogenatomer?